# Related-Market Ranking Analysis

This notebook compares ChaosWing's lexical baseline against the context-aware reranker and inspects human usefulness labels, agreement, and contested candidates.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = Path("../ml_data") if Path("../ml_data").exists() else Path("ml_data")

def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    return pd.DataFrame(rows)

ranking_df = load_jsonl(DATA_DIR / "related_market_ranking_examples.jsonl")
judgments_df = load_jsonl(DATA_DIR / "related_market_judgments.jsonl")
usefulness_df = load_jsonl(DATA_DIR / "related_market_usefulness_examples.jsonl")
experiments_df = load_jsonl(DATA_DIR / "experiments.jsonl")

ranking_df.head()

In [ ]:
candidate_rows = []
for _, row in ranking_df.iterrows():
    for candidate in row.get("candidates", []):
        candidate_rows.append({
            "event_title": row.get("event_title"),
            "candidate_title": candidate.get("title"),
            "relevance": candidate.get("relevance"),
            "baseline_score": candidate.get("baseline_score"),
            "model_score": candidate.get("model_score"),
        })

candidates_df = pd.DataFrame(candidate_rows)
candidates_df.head()

In [ ]:
if not candidates_df.empty:
    score_summary = candidates_df[["baseline_score", "model_score", "relevance"]].describe()
    display(score_summary)

if not usefulness_df.empty:
    display(usefulness_df.head())

if not judgments_df.empty and "review_state" in judgments_df:
    judgments_df["review_state"].value_counts(dropna=False)


In [ ]:
ranking_runs = experiments_df[experiments_df.get("task_type", pd.Series(dtype=str)) == "related_market_ranking"].copy()
usefulness_runs = experiments_df[experiments_df.get("task_type", pd.Series(dtype=str)) == "related_market_usefulness"].copy()
display(ranking_runs[["title", "dataset_version", "metrics"]].tail())
display(usefulness_runs[["title", "dataset_version", "metrics"]].tail())


## Findings

- Record whether the context-aware reranker improves on lexical overlap for the current dataset.
- Summarize where judged usefulness agrees with the silver benchmark and where it diverges.

## Limitations

- Note where label coverage is still thin.
- Call out contested candidates and any ambiguity between topical similarity and trading usefulness.